In [1]:
"# Real-Time Facial Emotion Classification",
"This notebook trains a CNN model to classify facial emotions (happy, sad, neutral) and performs real-time inference using the PC's webcam."

"This notebook trains a CNN model to classify facial emotions (happy, sad, neutral) and performs real-time inference using the PC's webcam."

In [2]:
"## 1. Setup and Imports"
import numpy as np
import cv2
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
train_dir = './data/train'
test_dir = './data/test'
model_path = './models/emotion_model.h5'
cascade_path = './haarcascade_frontalface_default.xml'
img_size = (48, 48)
emotions = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']  # 7 classes (alphabetical order)

os.makedirs('./models', exist_ok=True)

In [3]:
"## 2. Train the Model\n",
"Run this cell to train the CNN. This may take 1-2 hours on a CPU. Adjust `epochs` if needed."

"# Data generators\n",
  # Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
test_datagen = ImageDataGenerator(rescale=1./255)

# Load all 7 classes (no filtering)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    color_mode='grayscale',
    batch_size=32,
    class_mode='categorical'
    # No 'classes' parameter: loads all folders (7 classes)
)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    color_mode='grayscale',
    batch_size=32,
    class_mode='categorical'
    # No 'classes' parameter: loads all folders (7 classes)
)

# Verify number of classes
print(f"Training classes: {train_generator.class_indices}")
print(f"Testing classes: {test_generator.class_indices}")
if len(train_generator.class_indices) != 7:
    raise ValueError(f"Expected 7 classes in train_generator, found {len(train_generator.class_indices)}")
if len(test_generator.class_indices) != 7:
    raise ValueError(f"Expected 7 classes in test_generator, found {len(test_generator.class_indices)}")

# Verify class order matches emotions list
expected_indices = {emotion: idx for idx, emotion in enumerate(emotions)}
if train_generator.class_indices != expected_indices:
    print("Warning: Class indices may not match emotions list. Adjust emotions order if needed.")

# Model architecture
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(48, 48, 1)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(7, activation='softmax')  # 7 classes: angry, disgust, fear, happy, sad, surprise, neutral
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train
history = model.fit(
    train_generator,
    steps_per_epoch=max(1, train_generator.samples // 32),
    epochs=20,
    validation_data=test_generator,
    validation_steps=max(1, test_generator.samples // 32)
)

# Save model
model.save(model_path)
print('Model trained and saved.')

Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.
Training classes: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}
Testing classes: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}


C:\Users\INDULA MADHUBHASHANA\anaconda3\envs\py310_env\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\INDULA MADHUBHASHANA\anaconda3\envs\py310_env\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
897/897 ━━━━━━━━━━━━━━━━━━━━ 575s 640ms/step - accuracy: 0.2558 - loss: 1.7972 - val_accuracy: 0.3105 - val_loss: 1.6861
Epoch 2/20
  1/897 ━━━━━━━━━━━━━━━━━━━━ 19s 22ms/step - accuracy: 0.3438 - loss: 1.7568

C:\Users\INDULA MADHUBHASHANA\anaconda3\envs\py310_env\lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


897/897 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.3438 - loss: 1.7568 - val_accuracy: 0.3100 - val_loss: 1.6872
Epoch 3/20
897/897 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.3331 - loss: 1.6673 - val_accuracy: 0.4482 - val_loss: 1.4714
Epoch 4/20
897/897 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.4062 - loss: 1.6194 - val_accuracy: 0.4425 - val_loss: 1.4789
Epoch 5/20
897/897 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.4102 - loss: 1.5221 - val_accuracy: 0.4794 - val_loss: 1.3519
Epoch 6/20
897/897 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5000 - loss: 1.3757 - val_accuracy: 0.4763 - val_loss: 1.3557
Epoch 7/20
897/897 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.4477 - loss: 1.4344 - val_accuracy: 0.4874 - val_loss: 1.3138
Epoch 8/20
897/897 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6562 - loss: 1.0657 - val_accuracy: 0.4888 - val_loss: 1.3200
Epoch 9/20
897/897 ━━━━━━━━━━━━━━━━━━━━ 41s 46ms/step - accuracy: 0.4712 - loss: 1.3818 - val_accuracy: 0.514

Model trained and saved.


In [4]:
"## 3. Real-Time Inference\n",
"Run this cell to start the webcam feed. Press 'q' to stop. Ensure the model and Haar cascade files are present."

from tensorflow.keras.models import load_model

# Load model and cascade
model = load_model(model_path)
face_cascade = cv2.CascadeClassifier(cascade_path)

# Webcam setup
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        print('Failed to capture video.')
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    for (x, y, w, h) in faces:
        roi_gray = gray[y:y+h, x:x+w]
        roi_resized = cv2.resize(roi_gray, (48, 48))
        roi_normalized = roi_resized / 255.0
        roi_reshaped = np.reshape(roi_normalized, (1, 48, 48, 1))

        prediction = model.predict(roi_reshaped)
        emotion_idx = np.argmax(prediction)
        emotion_label = emotions[emotion_idx]  # Now uses 7-class emotions list

        # Draw rectangle and label
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(frame, emotion_label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)

    cv2.imshow('Emotion Classifier', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━